# Four-way B / S / R / P Router — RF and XGB

Same experiment structure as the binary notebooks: `strat_y` and `strat_y_K`, RF/XGB, original-row 80/20 split, no target expansion, exact cross-branch ties excluded/audited, test + entire metrics, confusion matrices, leakage checks, slide tables, and CSV exports.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier
REPO_ROOT=Path.cwd(); sys.path.insert(0,str(REPO_ROOT)) if str(REPO_ROOT) not in sys.path else None
from restructured_pipeline.src.features import ALL_ALGOS, ROUTER_CLASSB_ALGOS, ROUTER_CLASSS_ALGOS, ROUTER_CLASSR_ALGOS, ROUTER_CLASSP_ALGOS, load_excel_data, process_kcore_list_to_statistics, post_process_data
from restructured_pipeline.src.data_splits import _train_test_split_k_aware
from restructured_pipeline.src.train import load_yaml_config
PROJECT_CONFIG=Path("restructured_pipeline/configs/project.yaml"); MODEL_CONFIG=Path("restructured_pipeline/configs/models_scheme1.yaml")
OUTPUT_DIR=Path("restructured_pipeline/outputs/reports/four_way_BSRP_classifier"); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
project=load_yaml_config(PROJECT_CONFIG); model_cfg=load_yaml_config(MODEL_CONFIG)
FEATURE_COLS=list(project["features"]["feature_cols"]); TEST_SIZE=project["split_rules"]["defaults"]["test_size"]; RANDOM_STATE=project["split_rules"]["defaults"]["random_state"]
CLASS_ORDER=["B","S","R","P"]; BRANCH_ALGOS={"B":list(ROUTER_CLASSB_ALGOS),"S":list(ROUTER_CLASSS_ALGOS),"R":list(ROUTER_CLASSR_ALGOS),"P":list(ROUTER_CLASSP_ALGOS)}
assert set().union(*map(set,BRANCH_ALGOS.values()))==set(ALL_ALGOS); assert sum(map(len,BRANCH_ALGOS.values()))==24
print(BRANCH_ALGOS)

{'B': ['kPEX-B-bnb', 'kPEX-oriented-B-bnb', 'UMKP-B-bnb', 'UMKP-oriented-B-bnb', 'UMKP+-B-bnb', 'UMKP+-oriented-B-bnb'], 'S': ['kPEX-S-bnb', 'kPEX-oriented-S-bnb', 'UMKP-S-bnb', 'UMKP-oriented-S-bnb', 'UMKP+-S-bnb', 'UMKP+-oriented-S-bnb'], 'R': ['kPEX-R-bnb', 'kPEX-oriented-R-bnb', 'UMKP-R-bnb', 'UMKP-oriented-R-bnb', 'UMKP+-R-bnb', 'UMKP+-oriented-R-bnb'], 'P': ['kPEX-P-bnb', 'kPEX-oriented-P-bnb', 'UMKP-P-bnb', 'UMKP-oriented-P-bnb', 'UMKP+-P-bnb', 'UMKP+-oriented-P-bnb']}


## 1. Load and preprocess canonical data

In [2]:
d=project["data"]
df=load_excel_data(d["file_path"],timeout_replacement=d["timeout_replacement"],y_cols=d["raw_target_columns"],sep=d["sep"],quotechar=d["quotechar"],engine=d["engine"],rename_columns=d["rename_columns"])
if "kcore_list" in df.columns and any(c not in df.columns for c in FEATURE_COLS): process_kcore_list_to_statistics(df,keep_col=False)
df=post_process_data(df)
missing=[c for c in ["dataset","k_value",*FEATURE_COLS,*ALL_ALGOS] if c not in df.columns]
if missing: raise ValueError(f"Missing columns: {missing}")
print("Original rows:",len(df)); display(df[["dataset","k_value",*FEATURE_COLS]].head())

Original rows: 4279


,dataset,k_value,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize
0,12month1,20,20,52328,1585220,30.294,9214,57,19,39,16785.102564,52328,1030,14354.722410
1,130bit,8,8,584,6059,10.375,322,12,8,5,564.400000,584,509,28.443628
2,130bit,9,9,584,6059,10.375,322,12,8,5,564.400000,584,509,28.443628
3,130bit,10,10,583,6051,10.379,321,12,9,4,559.500000,583,509,29.853810
4,144,4,4,144319,1072349,7.430,26,9,7,3,140266.333333,144319,132943,5188.210310


## 2. Build B/S/R/P target and exclude exact cross-branch ties
A tie within one branch is harmless. A row is excluded only when two or more branches share the exact minimum branch runtime.

In [3]:
records=[]
for idx,row in df.iterrows():
    mins={b:min(float(row[a]) for a in aa) for b,aa in BRANCH_ALGOS.items()}; best=min(mins.values()); winners=[b for b,v in mins.items() if v==best]
    records.append({"source_row_index":idx,"dataset":row["dataset"],"k_value":row["k_value"],**{f"{b}_best_runtime":mins[b] for b in CLASS_ORDER},"winning_branches":"|".join(winners),"n_winning_branches":len(winners),"target":winners[0] if len(winners)==1 else pd.NA})
target_audit=pd.DataFrame(records); tie_audit=target_audit[target_audit.n_winning_branches>1].copy(); usable=target_audit[target_audit.n_winning_branches==1]
work=df.loc[usable.source_row_index].copy(); work["target"]=usable.set_index("source_row_index").loc[work.index,"target"]
audit_summary=pd.DataFrame([{"total_original_rows":len(df),"usable_unambiguous_rows":len(work),"exact_cross_branch_ties":len(tie_audit),"tie_percent":100*len(tie_audit)/len(df),**{f"{b}_rows":int(work.target.eq(b).sum()) for b in CLASS_ORDER}}])
display(audit_summary); display(tie_audit.head(20)); audit_summary.to_csv(OUTPUT_DIR/"target_audit_summary.csv",index=False); tie_audit.to_csv(OUTPUT_DIR/"exact_cross_branch_ties.csv",index=False)

,total_original_rows,usable_unambiguous_rows,exact_cross_branch_ties,tie_percent,B_rows,S_rows,R_rows,P_rows
0,4279,3767,512,11.965412,1560,765,360,1082


,source_row_index,dataset,k_value,B_best_runtime,S_best_runtime,R_best_runtime,P_best_runtime,winning_branches,n_winning_branches,target
65,65,actor-collaboration,6,0.0011,0.0011,0.0011,0.0012,B|S|R,3,<NA>
66,66,actor-collaboration,7,0.0014,0.0014,0.0014,0.0014,B|S|R|P,4,<NA>
67,67,actor-collaboration,8,0.0016,0.0016,0.0017,0.0016,B|S|P,3,<NA>
68,68,actor-collaboration,9,0.0018,0.0019,0.0018,0.0019,B|R,2,<NA>
69,69,actor-collaboration,10,0.0021,0.0021,0.0022,0.0021,B|S|P,3,<NA>
135,135,af_5_k101,6,0.2916,0.2916,0.3000,0.2992,B|S,2,<NA>
163,163,aff-wiki-wordbypage,3,0.0035,0.0035,0.0035,0.0035,B|S|R|P,4,<NA>
164,164,aff-wiki-wordbypage,4,0.0036,0.0035,0.0035,0.0035,S|R|P,3,<NA>
167,167,aff-wiki-wordbypage,7,0.0000,0.0000,0.0000,0.0000,B|S|R|P,4,<NA>
168,168,aff-wiki-wordbypage,8,0.0000,0.0000,0.0000,0.0000,B|S|R|P,4,<NA>


## 3. Original-row `strat_y` and `strat_y_K` splitting — no target expansion

In [4]:
def make_split(frame,mode):
    X=frame[FEATURE_COLS].copy(); y=frame.target.copy()
    if mode=="strat_y": Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=TEST_SIZE,random_state=RANDOM_STATE,stratify=y)
    else:
        labels=y.astype(str)+"_k"+frame.k_value.astype(str)
        Xtr,Xte,ytr,yte=_train_test_split_k_aware(X,y,test_size=TEST_SIZE,random_state=RANDOM_STATE,stratify_labels=labels,context_label="four_way_BSRP_strat_y_K")
    if set(Xtr.index)&set(Xte.index): raise ValueError("Train/test leakage")
    if Xte.index.duplicated().any(): raise ValueError("Duplicate test indices")
    return Xtr,Xte,ytr,yte
splits={m:make_split(work,m) for m in ("strat_y","strat_y_K")}
rows=[]
for m,(Xtr,Xte,ytr,yte) in splits.items(): rows.append({"split_mode":m,"train_rows":len(Xtr),"test_rows":len(Xte),"overlap":len(set(Xtr.index)&set(Xte.index)),"duplicate_test_indices":int(Xte.index.duplicated().sum()),**{f"train_{c}":int(ytr.eq(c).sum()) for c in CLASS_ORDER},**{f"test_{c}":int(yte.eq(c).sum()) for c in CLASS_ORDER}})
split_audit=pd.DataFrame(rows); display(split_audit); split_audit.to_csv(OUTPUT_DIR/"split_audit.csv",index=False)

four_way_BSRP_strat_y_K: forced 0 row(s) into training before the K-aware split.
four_way_BSRP_strat_y_K: completed true target+K stratification on 44 combined class(es).
four_way_BSRP_strat_y_K: final train/test sizes = 3013/754.


,split_mode,train_rows,test_rows,overlap,duplicate_test_indices,train_B,train_S,train_R,train_P,test_B,test_S,test_R,test_P
0,strat_y,3013,754,0,0,1248,612,288,865,312,153,72,217
1,strat_y_K,3013,754,0,0,1247,612,289,865,313,153,71,217


## 4. RF/XGB configuration

In [5]:
def cfg(family,mode):
    prefix="rf" if family=="random_forest" else "xgb"; return dict(model_cfg["scheme1"][family]["variants"][f"{prefix}_{mode}"]["hyperparameters"])
def build_rf(mode): return RandomForestClassifier(**cfg("random_forest",mode))
def build_xgb(mode):
    p=cfg("xgboost",mode); p.update(objective="multi:softprob",num_class=4); return XGBClassifier(**p)
le=LabelEncoder(); le.fit(CLASS_ORDER); print(dict(zip(le.classes_,le.transform(le.classes_))))

{'B': 0, 'P': 1, 'R': 2, 'S': 3}


## 5. Metrics and confusion matrices

In [6]:
def metric_row(y,p,mode,model,scope):
    pm,rm,fm,_=precision_recall_fscore_support(y,p,labels=CLASS_ORDER,average="macro",zero_division=0); pw,rw,fw,_=precision_recall_fscore_support(y,p,labels=CLASS_ORDER,average="weighted",zero_division=0); pc,rc,fc,sc=precision_recall_fscore_support(y,p,labels=CLASS_ORDER,average=None,zero_division=0)
    z={"split_mode":mode,"model":model,"scope":scope,"rows":len(y),"accuracy":accuracy_score(y,p),"macro_precision":pm,"macro_recall":rm,"macro_f1":fm,"weighted_precision":pw,"weighted_recall":rw,"weighted_f1":fw}
    for i,c in enumerate(CLASS_ORDER): z.update({f"{c}_precision":pc[i],f"{c}_recall":rc[i],f"{c}_f1":fc[i],f"{c}_support":int(sc[i])})
    return z
def plot_cm(y,p,mode,model,scope):
    cm=confusion_matrix(y,p,labels=CLASS_ORDER); fig,ax=plt.subplots(figsize=(6,5)); ConfusionMatrixDisplay(cm,display_labels=CLASS_ORDER).plot(ax=ax,values_format="d"); ax.set_title(f"B/S/R/P - {model} - {mode} - {scope}"); fig.tight_layout(); fig.savefig(OUTPUT_DIR/f"confusion_{model}_{mode}_{scope}.png",dpi=180,bbox_inches="tight"); plt.show()

## 6. Train four models; evaluate held-out test and entire usable data

In [7]:
metric_rows=[]; pred_frames=[]
for mode,(Xtr,Xte,ytr,yte) in splits.items():
    for name in ("RF","XGB"):
        if name=="RF": model=build_rf(mode); model.fit(Xtr,ytr); predict=lambda X:model.predict(X)
        else: model=build_xgb(mode); model.fit(Xtr,le.transform(ytr)); predict=lambda X:le.inverse_transform(model.predict(X).astype(int))
        for scope,Xe,ye in (("test_data",Xte,yte),("entire_data",work[FEATURE_COLS],work.target)):
            yp=predict(Xe); metric_rows.append(metric_row(ye,yp,mode,name,scope)); plot_cm(ye,yp,mode,name,scope)
            q=work.loc[Xe.index,["dataset","k_value"]].copy(); q["source_row_index"]=Xe.index; q["split_mode"]=mode; q["model"]=name; q["scope"]=scope; q["true_target"]=np.asarray(ye); q["predicted_target"]=np.asarray(yp); q["correct"]=q.true_target.eq(q.predicted_target); pred_frames.append(q)
metrics=pd.DataFrame(metric_rows); predictions=pd.concat(pred_frames,ignore_index=True); display(metrics.round(4)); metrics.to_csv(OUTPUT_DIR/"metrics_all.csv",index=False); predictions.to_csv(OUTPUT_DIR/"predictions_all.csv",index=False)
for mode,(Xtr,Xte,_,_) in splits.items(): pd.DataFrame({"source_row_index":Xtr.index}).to_csv(OUTPUT_DIR/f"train_indices_{mode}.csv",index=False); pd.DataFrame({"source_row_index":Xte.index}).to_csv(OUTPUT_DIR/f"test_indices_{mode}.csv",index=False)

C:\Users\n1a9m\AppData\Local\Temp\ipykernel_1144\218164526.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  cm=confusion_matrix(y,p,labels=CLASS_ORDER); fig,ax=plt.subplots(figsize=(6,5)); ConfusionMatrixDisplay(cm,display_labels=CLASS_ORDER).plot(ax=ax,values_format="d"); ax.set_title(f"B/S/R/P - {model} - {mode} - {scope}"); fig.tight_layout(); fig.savefig(OUTPUT_DIR/f"confusion_{model}_{mode}_{scope}.png",dpi=180,bbox_inches="tight"); plt.show()
C:\Users\n1a9m\AppData\Local\Temp\ipykernel_1144\218164526.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  cm=confusion_matrix(y,p,labels=CLASS_ORDER); fig,ax=plt.subplots(figsize=(6,5)); ConfusionMatrixDisplay(cm,display_labels=CLASS_ORDER).plot(ax=ax,values_format="d"); ax.set_title(f"B/S/R/P - {model} - {mode} - {scope}"); fig.tight_layout(); fig.savefig(OUTPUT_DIR/f"confusion_{model}_{mode}_{scope}.png",dpi=180,bbox_inches="tight"); plt.show()
c:\Users\n1a9m\Anaconda3_\Li

,split_mode,model,scope,rows,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,...,S_f1,S_support,R_precision,R_recall,R_f1,R_support,P_precision,P_recall,P_f1,P_support
0,strat_y,RF,test_data,754,0.5650,0.5323,0.5241,0.5278,0.5646,0.5650,...,0.6689,153,0.3385,0.3056,0.3212,72,0.5110,0.5346,0.5225,217
1,strat_y,RF,entire_data,3767,0.8601,0.8415,0.8591,0.8495,0.8619,0.8601,...,0.8883,765,0.7463,0.8417,0.7911,360,0.8425,0.8457,0.8441,1082
2,strat_y,XGB,test_data,754,0.5517,0.5316,0.5101,0.5171,0.5491,0.5517,...,0.6580,153,0.4167,0.2778,0.3333,72,0.4746,0.5161,0.4945,217
3,strat_y,XGB,entire_data,3767,0.8880,0.8901,0.8757,0.8823,0.8884,0.8880,...,0.9149,765,0.8926,0.8083,0.8484,360,0.8538,0.8854,0.8693,1082
4,strat_y_K,RF,test_data,754,0.5557,0.5085,0.5015,0.5046,0.5551,0.5557,...,0.5529,153,0.3134,0.2958,0.3043,71,0.5174,0.5484,0.5324,217
5,strat_y_K,RF,entire_data,3767,0.8914,0.8791,0.8846,0.8818,0.8919,0.8914,...,0.8994,765,0.8235,0.8556,0.8392,360,0.8752,0.8882,0.8817,1082
6,strat_y_K,XGB,test_data,754,0.5650,0.5221,0.5060,0.5126,0.5603,0.5650,...,0.5597,153,0.3509,0.2817,0.3125,71,0.5475,0.5576,0.5525,217
7,strat_y_K,XGB,entire_data,3767,0.8938,0.8909,0.8807,0.8856,0.8938,0.8938,...,0.8992,765,0.8696,0.8333,0.8511,360,0.8856,0.8872,0.8864,1082


## 7. Slide-ready held-out table
Use this as the main result table.

In [8]:
cols=["split_mode","model","accuracy","macro_f1","weighted_f1"]+[f"{c}_{m}" for c in CLASS_ORDER for m in ("precision","recall","f1")]
slide=metrics.loc[metrics.scope.eq("test_data"),cols].copy(); nums=slide.select_dtypes(include="number").columns; slide[nums]=(100*slide[nums]).round(2); display(slide); slide.to_csv(OUTPUT_DIR/"slide_table_test_metrics.csv",index=False)

,split_mode,model,accuracy,macro_f1,weighted_f1,B_precision,B_recall,B_f1,S_precision,S_recall,S_f1,R_precision,R_recall,R_f1,P_precision,P_recall,P_f1
0,strat_y,RF,56.50,52.78,56.45,59.49,60.26,59.87,68.49,65.36,66.89,33.85,30.56,32.12,51.10,53.46,52.25
2,strat_y,XGB,55.17,51.71,54.88,57.91,58.65,58.28,65.58,66.01,65.80,41.67,27.78,33.33,47.46,51.61,49.45
4,strat_y_K,RF,55.57,50.46,55.50,62.46,63.26,62.86,57.86,52.94,55.29,31.34,29.58,30.43,51.74,54.84,53.24
6,strat_y_K,XGB,56.50,51.26,56.17,60.42,64.86,62.56,58.57,53.59,55.97,35.09,28.17,31.25,54.75,55.76,55.25


## 8. Slide-ready test vs entire table
Entire-data metrics include training rows and should be presented as descriptive fit metrics.

In [9]:
compact=metrics[["split_mode","model","scope","accuracy","macro_f1","weighted_f1"]].copy(); compact[["accuracy","macro_f1","weighted_f1"]]=(100*compact[["accuracy","macro_f1","weighted_f1"]]).round(2); display(compact); compact.to_csv(OUTPUT_DIR/"slide_table_test_vs_entire.csv",index=False)

,split_mode,model,scope,accuracy,macro_f1,weighted_f1
0,strat_y,RF,test_data,56.50,52.78,56.45
1,strat_y,RF,entire_data,86.01,84.95,86.06
2,strat_y,XGB,test_data,55.17,51.71,54.88
3,strat_y,XGB,entire_data,88.80,88.23,88.79
4,strat_y_K,RF,test_data,55.57,50.46,55.50
5,strat_y_K,RF,entire_data,89.14,88.18,89.16
6,strat_y_K,XGB,test_data,56.50,51.26,56.17
7,strat_y_K,XGB,entire_data,89.38,88.56,89.37


## 9. Generalization-gap table

In [10]:
gap=metrics.pivot(index=["split_mode","model"],columns="scope",values=["accuracy","weighted_f1"]); gap.columns=["_".join(c) for c in gap.columns]; gap=gap.reset_index(); gap["accuracy_gap_entire_minus_test"]=gap.accuracy_entire_data-gap.accuracy_test_data; gap["weighted_f1_gap_entire_minus_test"]=gap.weighted_f1_entire_data-gap.weighted_f1_test_data
for c in [x for x in gap if "accuracy" in x or "weighted_f1" in x]: gap[c]=(100*gap[c]).round(2)
display(gap); gap.to_csv(OUTPUT_DIR/"slide_table_generalization_gap.csv",index=False)

,split_mode,model,accuracy_entire_data,accuracy_test_data,weighted_f1_entire_data,weighted_f1_test_data,accuracy_gap_entire_minus_test,weighted_f1_gap_entire_minus_test
0,strat_y,RF,86.01,56.50,86.06,56.45,29.51,29.61
1,strat_y,XGB,88.80,55.17,88.79,54.88,33.63,33.91
2,strat_y_K,RF,89.14,55.57,89.16,55.50,33.57,33.66
3,strat_y_K,XGB,89.38,56.50,89.37,56.17,32.88,33.20


## Slide interpretation checklist
Prioritize held-out Accuracy, Weighted F1, Macro F1, B/S/R/P recall and the held-out confusion matrix. Compare RF vs XGB and `strat_y` vs `strat_y_K`. Report excluded exact cross-branch ties. Treat entire-data metrics as fit diagnostics because they include training rows.

## Wrong predictions and important routing failures

This section separates **classification errors** from **runtime-important routing errors**.

For each prediction we report the number of evaluated datapoints, correct predictions, wrong predictions, and wrong-prediction rate. A wrong routing decision is counted as an **important failure** only when:

1. the true global-optimum runtime is **greater than 5 seconds**, and
2. the best runtime available inside the **predicted branch/family** is **greater than 1.2× the global optimum**.

This evaluates the runtime consequence of the router decision. It does **not** assume that the downstream within-family algorithm selector necessarily chooses that family's optimum; therefore it is an optimistic routing-level failure analysis.


In [11]:
IMPORTANT_RUNTIME_SECONDS = 5.0
IMPORTANT_RATIO_THRESHOLD = 1.20

analysis_predictions = predictions.rename(columns={
    "scope": "evaluation_scope",
    "true_target": "true_label",
    "predicted_target": "predicted_label",
}).copy()
analysis_predictions["true_label"] = analysis_predictions["true_label"].astype(str)
analysis_predictions["predicted_label"] = analysis_predictions["predicted_label"].astype(str)

runtime_source = work

def family_best_runtime(row, label):
    if label not in BRANCH_ALGOS:
        raise ValueError(f"Unknown B/S/R/P label: {label!r}")
    return float(row[BRANCH_ALGOS[label]].min())

def attach_runtime_consequence(r):
    src = runtime_source.loc[int(r["source_row_index"])]
    global_best = float(src[list(ALL_ALGOS)].min())
    predicted_best = family_best_runtime(src, r["predicted_label"])
    return pd.Series({"global_fastest_runtime": global_best, "predicted_branch_best_runtime": predicted_best})

analysis_predictions[["global_fastest_runtime", "predicted_branch_best_runtime"]] = analysis_predictions.apply(attach_runtime_consequence, axis=1)

analysis_predictions["wrong_prediction"] = analysis_predictions["true_label"].ne(analysis_predictions["predicted_label"])
analysis_predictions["routing_time_ratio"] = analysis_predictions["predicted_branch_best_runtime"] / analysis_predictions["global_fastest_runtime"]
analysis_predictions["important_failure"] = (
    analysis_predictions["wrong_prediction"]
    & analysis_predictions["global_fastest_runtime"].gt(IMPORTANT_RUNTIME_SECONDS)
    & analysis_predictions["routing_time_ratio"].gt(IMPORTANT_RATIO_THRESHOLD)
)

# Table 1: prediction population and ordinary classification errors.
wrong_prediction_table = (
    analysis_predictions.groupby(["split_mode", "model", "evaluation_scope"], dropna=False)
    .agg(
        total_datapoints=("source_row_index", "size"),
        correct_predictions=("wrong_prediction", lambda s: int((~s).sum())),
        wrong_predictions=("wrong_prediction", "sum"),
    )
    .reset_index()
)
wrong_prediction_table["wrong_predictions"] = wrong_prediction_table["wrong_predictions"].astype(int)
wrong_prediction_table["accuracy_percent"] = (
    100 * wrong_prediction_table["correct_predictions"] / wrong_prediction_table["total_datapoints"]
).round(2)
wrong_prediction_table["wrong_prediction_percent"] = (
    100 * wrong_prediction_table["wrong_predictions"] / wrong_prediction_table["total_datapoints"]
).round(2)

# Table 2: consequential subset of the wrong predictions.
important_failure_table = (
    analysis_predictions.groupby(["split_mode", "model", "evaluation_scope"], dropna=False)
    .agg(
        total_datapoints=("source_row_index", "size"),
        wrong_predictions=("wrong_prediction", "sum"),
        important_failures=("important_failure", "sum"),
    )
    .reset_index()
)
important_failure_table[["wrong_predictions", "important_failures"]] = important_failure_table[["wrong_predictions", "important_failures"]].astype(int)
important_failure_table["important_failures_percent_of_wrong"] = np.where(
    important_failure_table["wrong_predictions"].gt(0),
    100 * important_failure_table["important_failures"] / important_failure_table["wrong_predictions"],
    0.0,
).round(2)
important_failure_table["important_failures_percent_of_all"] = (
    100 * important_failure_table["important_failures"] / important_failure_table["total_datapoints"]
).round(2)

print("Wrong-prediction analysis")
display(wrong_prediction_table)
print("Important routing failures (>5 s global optimum and >1.2x routing ratio)")
display(important_failure_table)

wrong_prediction_table.to_csv(OUTPUT_DIR / "slide_table_wrong_predictions.csv", index=False)
important_failure_table.to_csv(OUTPUT_DIR / "slide_table_important_failures.csv", index=False)
analysis_predictions.to_csv(OUTPUT_DIR / "prediction_failure_analysis.csv", index=False)


Wrong-prediction analysis


,split_mode,model,evaluation_scope,total_datapoints,correct_predictions,wrong_predictions,accuracy_percent,wrong_prediction_percent
0,strat_y,RF,entire_data,3767,3240,527,86.01,13.99
1,strat_y,RF,test_data,754,426,328,56.50,43.50
2,strat_y,XGB,entire_data,3767,3345,422,88.80,11.20
3,strat_y,XGB,test_data,754,416,338,55.17,44.83
4,strat_y_K,RF,entire_data,3767,3358,409,89.14,10.86
5,strat_y_K,RF,test_data,754,419,335,55.57,44.43
6,strat_y_K,XGB,entire_data,3767,3367,400,89.38,10.62
7,strat_y_K,XGB,test_data,754,426,328,56.50,43.50


Important routing failures (>5 s global optimum and >1.2x routing ratio)


,split_mode,model,evaluation_scope,total_datapoints,wrong_predictions,important_failures,important_failures_percent_of_wrong,important_failures_percent_of_all
0,strat_y,RF,entire_data,3767,527,25,4.74,0.66
1,strat_y,RF,test_data,754,328,18,5.49,2.39
2,strat_y,XGB,entire_data,3767,422,21,4.98,0.56
3,strat_y,XGB,test_data,754,338,19,5.62,2.52
4,strat_y_K,RF,entire_data,3767,409,21,5.13,0.56
5,strat_y_K,RF,test_data,754,335,21,6.27,2.79
6,strat_y_K,XGB,entire_data,3767,400,22,5.50,0.58
7,strat_y_K,XGB,test_data,754,328,22,6.71,2.92
